# Pilot Notebook (T4) — Kurulum, EDA, Baseline, Küçük-Ölçek Eğitim

Bu notebook, Google Colab'da **T4 GPU** ile çalıştırılmak üzere tasarlanmıştır. Amacı, tüm
pipeline'ın (veri hazırlama -> EDA -> baseline değerlendirme -> LoRA eğitimi) küçük bir
veri alt kümesiyle **hatasız uçtan uca çalıştığını** doğrulamaktır (bir "duman testi").
Tam ölçekli eğitim için `01_full_training_a100.ipynb` kullanılır.

Çalıştırmadan önce Colab menüsünden: **Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU** seçili olmalıdır.

Hücreleri SIRAYLA çalıştırın.

## 1) Google Drive'ı bağla
Tüm kalıcı veri (ham/işlenmiş veri, checkpoint, log, değerlendirme sonuçları) Drive'da
tutulur; böylece Colab oturumu kapansa bile ilerleme kaybolmaz.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2) Repoyu klonla ve bağımlılıkları kur

In [5]:
import os

REPO_URL = "https://github.com/nidazeren/qwen2.5-vl.git"
REPO_DIR = "/content/qwen2.5-vl"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

Cloning into '/content/qwen2.5-vl'...
remote: Enumerating objects: 104, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 104 (delta 52), reused 85 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (104/104), 78.04 KiB | 2.89 MiB/s, done.
Resolving deltas: 100% (52/52), done.
/content/qwen2.5-vl


In [ ]:
!pip install -q -r requirements.txt

# hf_xet (HuggingFace'in hizlandirilmis "Xet" indirme sistemi) bu ortamda tekrar tekrar
# indirmeleri %85-99 civarinda sessizce tikanmaya sokuyor (dosya/agirlik indirmeleri
# saatlerce "Reconstructing (incomplete total...)" durumunda kaliyor). Standart, daha
# yavas ama GUVENILIR HTTP indirmesine dusmesi icin kaldiriyoruz.
!pip uninstall -y hf_xet -q

# torchao: Colab'in onceden kurdugu surum (0.10.0), peft'in bekledigi surumle
# (>0.16.0) UYUMSUZ. peft, LoRA hedef modullerini eslerken (embed_tokens dahil) her
# modul icin "bu torchao ile nicemlenmis mi?" kontrolu yapiyor; bu kontrolun KENDISI
# eski surumde ImportError firlatiyor (training/train_sft.py -> apply_lora() adiminda
# cokme). Projemiz torchao KULLANMIYOR (4-bit icin bitsandbytes kullaniliyor), bu
# yuzden en temiz cozum paketi tamamen kaldirmak.
!pip uninstall -y torchao -q

## 3) Ortam değişkenleri: PILOT_MODE ve Drive kök klasörü
`QWEN_OCR_PILOT_MODE=1`, `configs/config.py` içindeki `PILOT_MODE` bayrağını `True` yapar
(küçük veri alt kümesi, 1 epoch, 4-bit yükleme, küçük batch — T4'e uygun ayarlar).

In [7]:
import os, sys

os.environ["QWEN_OCR_PILOT_MODE"] = "1"
os.environ["QWEN_OCR_DRIVE_ROOT"] = "/content/drive/MyDrive/qwen25vl_turkish_ocr"
sys.path.insert(0, REPO_DIR)

from configs import config
config.ensure_directories()
print("PILOT_MODE:", config.PILOT_MODE)
print("DRIVE_ROOT:", config.DRIVE_ROOT)
print("Hedef kova boyutlari (pilot):", config.compute_bucket_target_sizes())

PILOT_MODE: True
DRIVE_ROOT: /content/drive/MyDrive/qwen25vl_turkish_ocr
Hedef kova boyutlari (pilot): {'printed_synthetic': 360, 'scene_text': 0, 'handwriting_synthetic': 189, 'smhd_english': 126, 'replay_ocr': 135, 'replay_general': 90}


## 3b) (İsteğe bağlı) RUN_NAME öncesi eski çıktıları arşivle
`configs/config.py`'ye eklenen `RUN_NAME` alt-klasörleme yapısı (bkz. ablation altyapısı
güncellemesi) checkpoint/log/eval çıktılarının konumunu değiştirdi. Daha önce bu notebook'u
çalıştırdıysanız, eski (RUN_NAME öncesi) çıktılar hâlâ Drive'da duruyor olabilir — yeni
kodu bozmazlar (farklı yoldalar) ama karışıklık yaratabilirler. Bu hücre onları **silmez**,
Drive'da bir `_archive_pre_run_name_*` klasörüne taşır. Temiz bir Drive ile başlamak
istemiyorsanız bu hücreyi atlayabilirsiniz.

In [ ]:
import shutil
from datetime import datetime

# RUN_NAME'e göre alt-klasörleme (bkz. configs/config.py) getirilmeden ÖNCE üretilmiş
# çıktılar farklı bir dizin yapısındaydı: checkpoints/{mode}/trainer_output (RUN_NAME
# alt klasörü OLMADAN), logs/{mode}/<event dosyaları doğrudan burada>,
# eval_outputs/{mode}/epoch_N.json + regression_report.json (runs/{RUN_NAME}/ OLMADAN).
# Bu hücre bunları SİLMEZ, geri alınabilir şekilde bir arşiv klasörüne TAŞIR --
# baseline.json ve onun checkpoints/baseline_*.jsonl önbelleği KORUNUR (hâlâ paylaşılan/
# geçerlidir, RUN_NAME'den etkilenmez, bkz. configs/config.py notu).
ARCHIVE_DIR = config.DRIVE_ROOT / f"_archive_pre_run_name_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OLD_CHECKPOINT_SUBDIRS = ["trainer_output", "final_adapter", "best_pareto_adapter"]

moved = []
for mode_tag in ["pilot", "full"]:
    ckpt_dir = config.DRIVE_ROOT / "checkpoints" / mode_tag
    if ckpt_dir.exists():
        for name in OLD_CHECKPOINT_SUBDIRS:
            old_path = ckpt_dir / name
            if old_path.exists() and old_path.is_dir():
                dest = ARCHIVE_DIR / "checkpoints" / mode_tag / name
                dest.parent.mkdir(parents=True, exist_ok=True)
                shutil.move(str(old_path), str(dest))
                moved.append(str(old_path))

    log_dir = config.DRIVE_ROOT / "logs" / mode_tag
    if log_dir.exists():
        for item in log_dir.iterdir():
            if item.is_file():  # RUN_NAME öncesi: event dosyaları doğrudan burada
                dest = ARCHIVE_DIR / "logs" / mode_tag / item.name
                dest.parent.mkdir(parents=True, exist_ok=True)
                shutil.move(str(item), str(dest))
                moved.append(str(item))

    eval_dir = config.DRIVE_ROOT / "eval_outputs" / mode_tag
    if eval_dir.exists():
        for pattern in ["epoch_*.json", "regression_report.json"]:
            for item in eval_dir.glob(pattern):
                dest = ARCHIVE_DIR / "eval_outputs" / mode_tag / item.name
                dest.parent.mkdir(parents=True, exist_ok=True)
                shutil.move(str(item), str(dest))
                moved.append(str(item))
        ckpt_cache = eval_dir / "checkpoints"
        if ckpt_cache.exists():
            for item in ckpt_cache.glob("epoch_*.jsonl"):
                dest = ARCHIVE_DIR / "eval_outputs" / mode_tag / "checkpoints" / item.name
                dest.parent.mkdir(parents=True, exist_ok=True)
                shutil.move(str(item), str(dest))
                moved.append(str(item))

if moved:
    print(f"{len(moved)} eski (RUN_NAME öncesi) öğe arşivlendi -> {ARCHIVE_DIR}")
    for m in moved:
        print(" ", m)
else:
    print("Taşınacak eski (RUN_NAME öncesi) çıktı bulunamadı -- temiz.")

## 4) (İsteğe bağlı) Kaggle API kimlik bilgisi — TS-TR için

`configs/config.py` içinde `USE_SCENE_TEXT = False` (VARSAYILAN) olduğu sürece bu
kaynak tamamen atlanır ve Kaggle'a hiç bağlanılmaz — **bu hücreyi ATLAYIP doğrudan
5. adıma geçebilirsiniz.**

TS-TR'yi (gerçek sahne metni) sonradan eklemek isterseniz: `configs/config.py`'de
`USE_SCENE_TEXT = True` yapıp GitHub'a push edin, Colab'da `git pull` çekin, sonra bu
hücreyi çalıştırın. İKİ yöntemden biriyle kimlik bilgisi sağlayabilirsiniz:

**Yöntem A — Colab Secrets (ÖNERİLEN, tarayıcı indirme sorunlarını atlar):**
1. kaggle.com > sağ üst profil > **Settings > API > Create New Token**.
2. Colab'ın sol kenar çubuğundaki **anahtar 🔑 simgesine** tıklayın.
3. `KAGGLE_USERNAME` ve `KAGGLE_KEY` adlarıyla iki gizli değer ekleyin, bu notebook için erişimi açın.

**Yöntem B — kaggle.json dosyası yükleme:**
Kaggle'dan indirdiğiniz `kaggle.json` dosyasını, hücre çalıştığında açılacak
"Dosya Seç" penceresinden yükleyin.

In [ ]:
import os, json, shutil, stat

if not config.USE_SCENE_TEXT:
    print("USE_SCENE_TEXT=False; Kaggle kimlik bilgisi gerekmiyor, hucre atlaniyor.")
else:
    kaggle_dir = os.path.expanduser("~/.kaggle")
    os.makedirs(kaggle_dir, exist_ok=True)
    target = os.path.join(kaggle_dir, "kaggle.json")
    drive_copy = str(config.DRIVE_ROOT / "kaggle.json")

    def _try_colab_secrets() -> bool:
        """Colab Secrets panelinde KAGGLE_USERNAME/KAGGLE_KEY varsa kaggle.json'i
        bunlardan uretir. Basarili olursa True doner."""
        try:
            from google.colab import userdata

            username = userdata.get("KAGGLE_USERNAME")
            key = userdata.get("KAGGLE_KEY")
        except Exception:
            return False
        if not username or not key:
            return False
        with open(target, "w", encoding="utf-8") as f:
            json.dump({"username": username, "key": key}, f)
        print("kaggle.json, Colab Secrets (KAGGLE_USERNAME/KAGGLE_KEY) kullanilarak olusturuldu.")
        return True

    if os.path.exists(drive_copy):
        shutil.copy(drive_copy, target)
        print("kaggle.json Drive'dan kopyalandi.")
    elif os.path.exists(target):
        print("kaggle.json zaten mevcut.")
    elif _try_colab_secrets():
        pass
    else:
        from google.colab import files
        print("Colab Secrets bulunamadi. Lutfen kaggle.json dosyanizi secin:")
        uploaded = files.upload()
        uploaded_name = next(iter(uploaded))
        shutil.move(uploaded_name, target)

    shutil.copy(target, drive_copy)  # sonraki oturumlar icin Drive'a da kopyala (bir daha sorulmasin diye)
    os.chmod(target, stat.S_IRUSR | stat.S_IWUSR)
    print("Kaggle kimlik bilgisi hazir:", target)

## 5) (Opsiyonel) SMHD el yazısı verisi
`configs/config.py` içinde `USE_SMHD = True` ise, `hiqmatNisa/SMHD` GitHub reposundaki
izin formunu doldurup indirdiğiniz veriyi şu klasöre yerleştirmelisiniz:
`config.SMHD_LOCAL_DIR` (varsayılan: `.../qwen25vl_turkish_ocr/manual_datasets/SMHD`).
Varsayılan `USE_SMHD=False` iken bu adımı atlayabilirsiniz; kod otomatik olarak
el yazısı payının tamamını `emredeveloper/turkish-ocr` kaynağına kaydırır.

In [8]:
print("USE_SMHD =", config.USE_SMHD)
print("Beklenen SMHD klasoru:", config.SMHD_LOCAL_DIR)

USE_SMHD = True
Beklenen SMHD klasoru: /content/drive/MyDrive/qwen25vl_turkish_ocr/manual_datasets/SMHD


## 6) Veri hazırlama (pilot alt küme)
Tüm ham kaynakları indirir/normalize eder; `PILOT_MODE=True` olduğundan her kaynaktan
yalnızca `config.PILOT_SAMPLES_PER_SOURCE` kadar örnek kullanılır (hızlı çalışsın diye).

In [11]:
!python data/prepare_datasets.py

[skip] printed_synthetic zaten diskte mevcut (/content/drive/MyDrive/qwen25vl_turkish_ocr/raw_data/printed_synthetic); yeniden indirilmiyor. Yeniden üretmek isterseniz bu klasörü silip tekrar çalıştırın.
[2/5] TS-TR atlandı (config.USE_SCENE_TEXT=False, Kaggle gerekmez).
      scene_text için 0 kayıt bulundu, diske yazılmadı.
[skip] handwriting_synthetic zaten diskte mevcut (/content/drive/MyDrive/qwen25vl_turkish_ocr/raw_data/handwriting_synthetic); yeniden indirilmiyor. Yeniden üretmek isterseniz bu klasörü silip tekrar çalıştırın.
[4/5] SMHD okunuyor (kaynak: /content/drive/MyDrive/qwen25vl_turkish_ocr/manual_datasets/SMHD)
      Yerel önbellek zaten mevcut, kopyalama atlanıyor: /tmp/qwen_ocr_local_cache/SMHD
      SMHD görselleri işleniyor:  45% 19/42 [00:03<00:04,  5.67it/s]      !! 0003.jpg okunamadı, atlanıyor: image file is truncated (48 bytes not processed)
      SMHD görselleri işleniyor: 100% 42/42 [00:07<00:00,  5.53it/s]
      -> 41 örnek yüklendi.
Saving the dataset (1/1 

In [10]:
!pip uninstall -y hf_xet -q

## 7) EDA: Tokenizer analizi ve görsel token sayısı
Tokenizer analizi, `ENABLE_EMBED_LORA` için bir öneri üretir (raporu okuyup kararı siz
verirsiniz). Görsel token EDA'sı, `MIN_PIXELS`/`MAX_PIXELS` seçiminize yardımcı olur.

In [9]:
!python analysis/tokenizer_analysis.py

[tokenizer_analysis] Tokenizer yükleniyor: Qwen/Qwen2.5-VL-3B-Instruct
config.json: 100% 1.37k/1.37k [00:00<00:00, 5.06MB/s]
tokenizer_config.json: 100% 5.70k/5.70k [00:00<00:00, 20.6MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 99.7MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 114MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 137MB/s]

=== Tek karakter tokenizasyonu ===
  'ç': 1 token -> ['Ã§']
  'ğ': 1 token -> ['ÄŁ']
  'ı': 1 token -> ['Ä±']
  'ö': 1 token -> ['Ã¶']
  'ş': 1 token -> ['ÅŁ']
  'ü': 1 token -> ['Ã¼']
  'Ç': 1 token -> ['Ãĩ']
  'Ğ': 1 token -> ['Äŀ']
  'İ': 1 token -> ['Ä°']
  'Ö': 1 token -> ['Ãĸ']
  'Ş': 1 token -> ['Åŀ']
  'Ü': 1 token -> ['Ãľ']

=== Kelime listesi karşılaştırması ===
  n_special_words: 172
  n_plain_words: 128
  special_words_tokens_per_char: 0.4025
  plain_words_tokens_per_char: 0.3407
  fragmentation_ratio: 1.1814

=== Parçalanma oranı: 1.1814 (eşik: 1.15) ===
ÖNERİ: Türkçe özel karakterler belirgin şekilde daha fazla parçalanıyor. co

In [10]:
!python analysis/vision_token_eda.py

[vision_token_eda] Processor yükleniyor: Qwen/Qwen2.5-VL-3B-Instruct
preprocessor_config.json: 100% 350/350 [00:00<00:00, 2.25MB/s]
chat_template.json: 100% 1.05k/1.05k [00:00<00:00, 4.65MB/s]
[vision_token_eda] 40 görsel üzerinde analiz yapılacak (fallback=False).

[dar_T4_dostu (128-512 blok)] min_pixels=100352, max_pixels=401408
  token sayısı -> min=132, medyan=238.0, ortalama=231.2, maks=322
  (config.MAX_SEQ_LENGTH=1024 ile karşılaştırın: görsel token + talimat + hedef metin token'ları bu sınırı aşmamalı)

[orta_varsayilan (256-768 blok, config.py)] min_pixels=200704, max_pixels=602112
  token sayısı -> min=264, medyan=308.0, ortalama=298.7, maks=322
  (config.MAX_SEQ_LENGTH=1024 ile karşılaştırın: görsel token + talimat + hedef metin token'ları bu sınırı aşmamalı)

[genis_resmi_varsayilan (256-1280 blok)] min_pixels=200704, max_pixels=1003520
  token sayısı -> min=264, medyan=308.0, ortalama=298.7, maks=322
  (config.MAX_SEQ_LENGTH=1024 ile karşılaştırın: görsel token + talimat 

In [3]:
%cd /content/qwen2.5-vl

[Errno 2] No such file or directory: '/content/qwen2.5-vl'
/content


## 8) Self-distillation replay verisi üretimi
Taban model (henüz LoRA uygulanmadan), OmniDocBench görselleri üzerinde çalıştırılıp
kendi çıktıları "replay" hedef metni olarak kaydedilir. Bu adım GPU kullanır ve pilot
modda bile birkaç dakika sürebilir.

In [12]:
!python data/replay_generation.py

[replay_generation] Hedef: replay_ocr=135, replay_general=90
[replay_generation] Taban model (LoRA UYGULANMADAN) yükleniyor...
[lora_setup] Seçilen dtype=torch.bfloat16, attn_implementation=sdpa, 4bit=True
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files: 100% 2/2 [00:00<00:00, 726.92it/s]
Download complete: :           |  0.00B            
Download complete: :           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B            
Loading weights: 100% 824/824 [01:37<00:00,  8.43it/s]
[replay_generation] replay_ocr: 100% 135/135 [1:00:14<00:00, 26.78s/it]
[replay_generation] replay_general: 100% 90/90 [31:23<00:00, 20.93s/it]
Saving the dataset (1/1 shards): 100% 135/135 [00:00<00:00, 468.96 examples/s]
      Kaydedildi: /content/drive/MyDrive/qwen25vl_turkish_ocr/raw_data/replay_ocr (135 örnek)
Saving the dataset (1/1 shards): 100% 90/90 [00:00<00:00, 460.63 examples/s]
      Kaydedildi: /content/drive/MyDrive/q

In [12]:
!pip install -q -r requirements.txt
!pip uninstall -y hf_xet -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 139.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 116.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 72.3 MB/s eta 0:00:00


## 9) Eğitim/Validation/Test A/Test B setlerini oluştur

In [13]:
!python data/build_chat_dataset.py

[build_chat_dataset] Hedef kova boyutları: {'printed_synthetic': 360, 'scene_text': 0, 'handwriting_synthetic': 189, 'smhd_english': 126, 'replay_ocr': 135, 'replay_general': 90}
Traceback (most recent call last):
  File "/content/qwen2.5-vl/data/build_chat_dataset.py", line 221, in <module>
    main()
  File "/content/qwen2.5-vl/data/build_chat_dataset.py", line 149, in main
    taken, pools[name] = _take(pools[name], n)
                         ^^^^^^^^^^^^^^^^^^^^^
  File "/content/qwen2.5-vl/data/build_chat_dataset.py", line 71, in _take
    return ds.select(range(n)), ds.select(range(n, len(ds)))
                                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/datasets/arrow_dataset.py", line 576, in wrapper
    out: Union["Dataset", "DatasetDict"] = func(self, *args, **kwargs)
                                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/datasets/fingerprint.py", line 468, in wrap

In [14]:
!cd /content/qwen2.5-vl && git pull
!python data/build_chat_dataset.py

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.03 KiB | 1.03 MiB/s, done.
From https://github.com/nidazeren/qwen2.5-vl
   da73a6c..72abf95  main       -> origin/main
Updating da73a6c..72abf95
Fast-forward
 data/build_chat_dataset.py | 14 ++++++++++++--
 1 file changed, 12 insertions(+), 2 deletions(-)
[build_chat_dataset] Hedef kova boyutları: {'printed_synthetic': 360, 'scene_text': 0, 'handwriting_synthetic': 189, 'smhd_english': 126, 'replay_ocr': 135, 'replay_general': 90}
Filter: 100% 233/233 [00:11<00:00, 21.06 examples/s]
Filter: 100% 233/233 [00:01<00:00, 130.87 examples/s]
      !! Havuzda 199 örnek var, 360 isteniyor; tekrarlı örnekleme yapılacak.
Map: 100% 495/495 [01:06<00:00,  7.48 examples/s]
Saving the dataset (1/1 shards): 100% 495/495 [00:00<00:00, 1513.72 examples/s]
[build_chat_dataset

## 10) Baseline değerlendirme
LoRA henüz uygulanmadığı için burada taban modelin Test A/B skorları ölçülür ve
`eval_outputs/baseline.json` olarak kaydedilir. (training/train_sft.py, bu dosya yoksa
kendisi de otomatik üretir; ama pilot akışında ayrıca burada da görmek isteyebilirsiniz.)

In [15]:
!python evaluation/evaluate.py --tag baseline

[lora_setup] Seçilen dtype=torch.bfloat16, attn_implementation=sdpa, 4bit=True
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files: 100% 2/2 [00:00<00:00, 928.56it/s]
Download complete: :           |  0.00B            
Download complete: :           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B            
Loading weights: 100% 824/824 [00:04<00:00, 165.00it/s]
[evaluate] 'baseline' etiketiyle değerlendirme başlıyor...
Test Seti A:   0% 0/157 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:944: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(
Test Seti A:  18% 28/157 [06:49<31:27, 14.63s/it]
Traceback (most recent call last):
  File "/content/qwen2.5-vl/evaluation/evaluate.py", line 121, in <module>
    main()
  File "/content/qwen2.5-vl/evaluation/evaluate.py", line 117, in main
    evalu

In [16]:
!cd /content/qwen2.5-vl && git pull
!python evaluation/evaluate.py --tag baseline

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.66 KiB | 1.66 MiB/s, done.
From https://github.com/nidazeren/qwen2.5-vl
   72abf95..9f8de63  main       -> origin/main
Updating 72abf95..9f8de63
Fast-forward
 evaluation/evaluate.py | 65 +++++++++++++++++++++++++++++++++++++++++---------
 1 file changed, 54 insertions(+), 11 deletions(-)
[lora_setup] Seçilen dtype=torch.bfloat16, attn_implementation=sdpa, 4bit=True
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files: 100% 2/2 [00:00<00:00, 911.71it/s]
Download complete: :           |  0.00B            
Download complete: :           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B            
Loading weights: 100% 824/824 [00:04<00:00, 166.25it/s]
[evaluate] 'baseline' etiketiyle değerlen

In [17]:
!cd /content/qwen2.5-vl && git pull
!python evaluation/evaluate.py --tag baseline

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 732 bytes | 732.00 KiB/s, done.
From https://github.com/nidazeren/qwen2.5-vl
   9f8de63..72088e0  main       -> origin/main
Updating 9f8de63..72088e0
Fast-forward
 evaluation/evaluate.py | 4 +++-
 1 file changed, 3 insertions(+), 1 deletion(-)
[lora_setup] Seçilen dtype=torch.bfloat16, attn_implementation=sdpa, 4bit=True
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files: 100% 2/2 [00:00<00:00, 817.36it/s]
Download complete: :           |  0.00B            
Download complete: :           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B            
Loading weights: 100% 824/824 [00:04<00:00, 167.85it/s]
[evaluate] 'baseline' etiketiyle değerlendirme başlıyor...
      Önceki bir çalıştırmada

## 11) Pilot LoRA eğitimi (T4, 4-bit + gradient checkpointing)
`PILOT_MODE=True` olduğundan: küçük veri, 1 epoch, düşük batch + yüksek gradient
accumulation, 4-bit yükleme. Bu adımın hatasız tamamlanması, A100'de tam eğitime
geçmeden önceki nihai doğrulamadır.

In [ ]:
!python training/train_sft.py

## 11b) Ablation altyapısının hızlı doğrulaması (smoke test)
Bu bölüm **gerçek bir ablation karşılaştırması değildir** — pilot verisi (~500 örnek,
1 epoch) böyle bir karşılaştırma için istatistiksel olarak çok küçüktür; asıl faz-faz
ablation taraması (`LR` → `layer coverage` → `attention/MLP` → `rank/alpha/dropout` →
`replay loss weighting`) yalnızca `01_full_training_a100.ipynb`'de, tam veriyle çalışır.

Burada yapılan, `configs/config.py`'ye eklenen yeni ablation parametrelerinin
(`LORA_LAYER_SCOPE`, `LORA_TARGET_SCOPE`, `ENABLE_WEIGHTED_LOSS`, isteğe bağlı olarak
`ENABLE_ONLINE_SELF_DISTILLATION`) ve drift takibinin (`training/lora_drift.py`) T4'te
**hatasız çalıştığını** doğrulamaktır — A100'de saatler sürecek tam ablation taramasına
geçmeden önceki son güvenlik kontrolü. Her koşum kendi `RUN_NAME`'i altında izole
çalışır; yukarıdaki "11) Pilot LoRA eğitimi" adımının (`RUN_NAME="default"`)
çıktısını EZMEZ.

In [ ]:
import os, sys, json, subprocess

def run_smoke_test(run_name: str, env_overrides: dict) -> bool:
    """Verilen ortam değişkeni override'larıyla training/train_sft.py'yi (pilot
    verisiyle, 1 epoch) çalıştırıp BAŞARIYLA (hatasız) tamamlanıp tamamlanmadığını
    döner. Amaç gerçek bir ablation karşılaştırması DEĞİL -- pilot verisi (~500 örnek,
    1 epoch) istatistiksel olarak anlamlı bir LR/layer-scope karşılaştırması için çok
    küçüktür (bkz. `01_full_training_a100.ipynb`'deki asıl ablation fazları). Burada
    yalnızca configs/config.py'ye eklenen yeni ablation parametrelerinin (layer scope,
    target scope, weighted loss, online distillation, drift tracking) kod yolunun
    T4'te HATASIZ ÇALIŞTIĞINI doğruluyoruz -- A100'de saatler süren tam ablation
    taramasına geçmeden önceki son güvenlik kontrolü."""
    env = os.environ.copy()
    env["QWEN_OCR_RUN_NAME"] = run_name
    for key, value in env_overrides.items():
        env[key] = str(value)

    print(f"\n{'='*80}\n[smoke-test] run_name={run_name!r} overrides={env_overrides}\n{'='*80}")
    result = subprocess.run([sys.executable, "training/train_sft.py"], env=env, cwd=REPO_DIR)
    ok = result.returncode == 0
    print(f"[smoke-test] run_name={run_name!r} -> {'OK' if ok else 'HATA'} (returncode={result.returncode})")
    return ok

In [ ]:
SMOKE_TEST_SCENARIOS = {
    "smoke_last12_attn_only": {
        "QWEN_OCR_LORA_LAYER_SCOPE": "last12",
        "QWEN_OCR_LORA_TARGET_SCOPE": "attn_only",
    },
    "smoke_weighted_loss": {
        "QWEN_OCR_ENABLE_WEIGHTED_LOSS": "1",
        "QWEN_OCR_LOSS_WEIGHT_REPLAY": "1.5",
    },
}

# Online self-distillation ikinci bir ~3B modeli belleğe yükler; T4'ün 16 GB'ında
# 4-bit + gradient checkpointing ile büyük olasılıkla sığar ama YAVAŞTIR (her batch'te
# ekstra bir forward geçişi). Denemek isterseniz True yapın.
RUN_ONLINE_DISTILL_SMOKE_TEST = False
if RUN_ONLINE_DISTILL_SMOKE_TEST:
    SMOKE_TEST_SCENARIOS["smoke_online_distill"] = {"QWEN_OCR_ENABLE_ONLINE_DISTILL": "1"}

smoke_test_results = {
    run_name: run_smoke_test(run_name, overrides)
    for run_name, overrides in SMOKE_TEST_SCENARIOS.items()
}

print("\n=== Smoke test özeti ===")
for run_name, ok in smoke_test_results.items():
    print(f"  {run_name}: {'OK' if ok else 'HATA'}")

In [ ]:
for run_name in SMOKE_TEST_SCENARIOS:
    run_dir = config.EVAL_OUTPUT_DIR / "runs" / run_name
    run_config_path = run_dir / "run_config.json"
    drift_path = run_dir / "lora_drift" / "drift_log.jsonl"
    pareto_path = run_dir / "pareto_summary.json"

    print(f"--- {run_name} ---")
    if run_config_path.exists():
        print("  run_config.json:", json.loads(run_config_path.read_text(encoding="utf-8")))
    else:
        print("  !! run_config.json bulunamadı")
    print("  drift_log.jsonl var mı:", drift_path.exists(), f"({drift_path})")
    print("  pareto_summary.json var mı:", pareto_path.exists(), f"({pareto_path})")
    print()

## 12) Sonuçları incele
`baseline.json` paylaşılan klasördedir (`config.EVAL_OUTPUT_DIR`); `epoch_1.json`,
`regression_report.json`, `eval_history.json`, `pareto_summary.json` ise koşuma özgüdür
(`config.EVAL_RUN_OUTPUT_DIR`, "11) Pilot LoRA eğitimi" adımı için varsayılan
`RUN_NAME="default"` altında). Pilot başarılıysa (hata vermeden tamamlandıysa)
`01_full_training_a100.ipynb`'ye geçebilirsiniz.

In [ ]:
import json

# baseline.json PAYLAŞILAN klasördedir (config.EVAL_OUTPUT_DIR) -- RUN_NAME'e göre AYRILMAZ
# (bkz. configs/config.py notu). epoch_1.json/regression_report.json/eval_history.json/
# pareto_summary.json ise koşuma özgüdür (config.EVAL_RUN_OUTPUT_DIR, RUN_NAME set
# edilmediyse varsayılan "default" altında).
baseline_path = config.EVAL_OUTPUT_DIR / "baseline.json"
print("--- baseline.json ---")
if baseline_path.exists():
    print(json.dumps(json.loads(baseline_path.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))
else:
    print("yok (bu normal olabilir)")

for name in ["epoch_1.json", "regression_report.json", "eval_history.json", "pareto_summary.json"]:
    path = config.EVAL_RUN_OUTPUT_DIR / name
    print(f"\n--- {name} ---")
    if path.exists():
        print(json.dumps(json.loads(path.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))
    else:
        print("yok (bu normal olabilir)")